In [11]:
!pip install webrtcvad librosa soundfile scipy

In [21]:
from google.colab import files

uploaded = files.upload()

Saving sample_noisy.wav to sample_noisy.wav
Saving sample_clean.wav to sample_clean.wav


In [23]:
# ==========================================================
# Background Noise Detector
# Energy-Based Signal-to-Noise Estimation
# ==========================================================

import librosa
import numpy as np
import json

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------

QUALITY_THRESHOLD = 0.06      # Average RMS threshold


# ----------------------------------------------------------
# Analyze Audio
# ----------------------------------------------------------

def analyze(file):

    # Load audio
    y, sr = librosa.load(file, sr=None)

    # RMS Energy
    rms = librosa.feature.rms(
        y=y,
        frame_length=2048,
        hop_length=512
    )[0]

    # Remove silent frames
    rms_nonzero = rms[rms > 0.001]

    if len(rms_nonzero) == 0:
        rms_nonzero = rms

    # Energy Features
    average_rms = np.mean(rms_nonzero)

    noise_floor = np.percentile(rms_nonzero,10)

    speech_level = np.percentile(rms_nonzero,90)

    # Estimated SNR
    snr = 20 * np.log10(
        (speech_level + 1e-8) /
        (noise_floor + 1e-8)
    )

    # Classification
    if average_rms >= QUALITY_THRESHOLD:
        flag = "clean"
    else:
        flag = "noisy"

    # Output
    result = {

        "file": file,

        "average_rms": round(float(average_rms),4),

        "noise_floor": round(float(noise_floor),4),

        "speech_level": round(float(speech_level),4),

        "snr_estimate_db": round(float(snr),2),

        "flag": flag
    }

    print(json.dumps(result,indent=4))


# ----------------------------------------------------------
# Test Files
# ----------------------------------------------------------

files = [

    "sample_clean.wav",

    "sample_noisy.wav"

]

for f in files:
    analyze(f)

{
    "file": "sample_clean.wav",
    "average_rms": 0.096,
    "noise_floor": 0.0137,
    "speech_level": 0.1848,
    "snr_estimate_db": 22.59,
    "flag": "clean"
}
{
    "file": "sample_noisy.wav",
    "average_rms": 0.0433,
    "noise_floor": 0.0017,
    "speech_level": 0.1447,
    "snr_estimate_db": 38.69,
    "flag": "noisy"
}
